# 05 — Robustesse : sous-déclaration, pondérations, intervalles de confiance

**Projet NutriHFr** · Niveau « attendu » de la consigne + discussion 6b (validation et robustesse)

Un résultat est **robuste** s'il change peu quand on modifie raisonnablement les hypothèses. On teste ici :
1. l'exclusion des **sous-déclarants** (critère de Goldberg) ;
2. l'hypothèse de niveau d'activité physique (PAL) utilisée par ce critère ;
3. l'utilisation ou non des **pondérations** ;
4. la méthode de calcul des **intervalles de confiance** (linéarisation vs bootstrap).

In [ ]:
# --- Configuration commune à tous les notebooks ------------------------------
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C
from nutrihfr.viz import set_style, savefig, PALETTE, SEX_COLORS
set_style()
if C.MODE == "synthetic" and not C.raw_dir("inca3").exists():   # 1re utilisation : données de démo
    import runpy; runpy.run_path(str(ROOT / "scripts" / "make_synthetic_data.py"), run_name="__main__")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
print("Mode des données :", C.MODE, "->", C.raw_dir("inca3"))

In [ ]:
from nutrihfr.pipeline import NutriPipeline
from nutrihfr.stats import describe_weighted, weighted_mean, weighted_se
from nutrihfr.intake import goldberg_cutoff, flag_underreporters, IntakeCalculator

res = NutriPipeline(verbose=False).run(save=False)
df, lines, NUTS = res["indiv"], res["lines"], res["nutrients"]

## 1. Repérer les sous-déclarants : critère de Goldberg

- **Métabolisme de base (BMR)** estimé par les équations de Schofield (OMS 1985) à partir du sexe, de l'âge et du poids.
- On compare l'apport énergétique déclaré (EI) au BMR. Une personne en équilibre pondéral dépense environ PAL × BMR (PAL ≈ 1,55 pour une activité légère à modérée).
- Le **seuil de Goldberg** (version Black, 2000) tient compte des erreurs de mesure et du **nombre de jours observés** : avec peu de jours, le seuil est plus bas (on tolère plus de variabilité).
- Si EI/BMR < seuil, l'apport déclaré est **incompatible** avec le maintien du poids → sous-déclarant probable.

In [ ]:
pd.DataFrame({"jours observés": [1, 2, 3, 7], "seuil EI/BMR": [goldberg_cutoff(d) for d in (1, 2, 3, 7)]}).set_index("jours observés")

In [ ]:
cls = df.dropna(subset=["sous_declarant"]).copy()
cls["sous_declarant"] = cls["sous_declarant"].astype(float)
print(f"Individus classables (poids connu) : {len(cls)} / {len(df)}")
prev = describe_weighted(cls, "sous_declarant", by=["sexe", "classe_age"])[["sexe", "classe_age", "n", "moyenne", "ic95_bas", "ic95_haut"]]
prev[["moyenne", "ic95_bas", "ic95_haut"]] *= 100
print(f"Prévalence pondérée globale : {100 * weighted_mean(cls.sous_declarant, cls.poids_enquete):.1f} %")
prev.rename(columns={"moyenne": "% sous-déclarants"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
for s in ["Homme", "Femme"]:
    d = cls[cls.sexe == s]
    ax.hist(d["ei_bmr"], bins=50, weights=d["poids_enquete"], density=True, alpha=0.4, color=SEX_COLORS[s], label=s)
ax.axvline(goldberg_cutoff(3), color="#0b0b0b", lw=1.5, ls="--")
ax.text(goldberg_cutoff(3), ax.get_ylim()[1] * 0.92, "  seuil de Goldberg (3 jours)", fontsize=9)
ax.set(title="Rapport apport énergétique / métabolisme de base", xlabel="EI / BMR", ylabel="densité")
ax.legend(); savefig(fig, "05_goldberg"); plt.show()

**Interprétation** — Les sous-déclarants sont typiquement plus nombreux chez les femmes et chez les personnes en surpoids. Attention : le critère repère les apports *implausiblement bas*, il ne corrige pas les sous-déclarations modérées ni les oublis sélectifs (grignotage, alcool).

## 2. Effet de l'exclusion des sous-déclarants sur les résultats

In [ ]:
plaus = cls[cls.sous_declarant == 0]
rows = []
for n in NUTS:
    a = weighted_mean(df[n], df.poids_enquete)
    b = weighted_mean(plaus[n], plaus.poids_enquete)
    rows.append({"nutriment": C.NUTRIENTS[n][0], "tous": a, "sans_sous_declarants": b, "variation_%": 100 * (b / a - 1)})
sens = pd.DataFrame(rows).set_index("nutriment")
sens.to_csv(C.TABLES / "sensibilite_sous_declarants.csv")
sens

In [ ]:
# Les parts de contribution (Q5) sont-elles stables ? (ratio des moyennes)
plaus_lines = lines[lines.id_indiv.isin(plaus.id_indiv)]
for n in ["proteines_g", "calcium_mg"]:
    a = IntakeCalculator.contributions(lines, df, n).set_index("groupe")["contribution_pct"]
    b = IntakeCalculator.contributions(plaus_lines, plaus, n).set_index("groupe")["contribution_pct"]
    t = pd.DataFrame({"tous": a, "sans_sous_declarants": b}).loc[a.index[:6]]
    print(f"\n{C.NUTRIENTS[n][0]} : mêmes 6 premiers groupes ? {set(a.index[:6]) == set(b.index[:6])}")
    display(t.round(1))

**Lecture** — Les apports **absolus** augmentent quand on retire les sous-déclarants (attendu), mais les **parts de contribution** et les **densités** bougent peu : les conclusions de Q4 et Q5 sont robustes, celles de Q1 dépendent davantage de la sous-déclaration.

## 3. Sensibilité à l'hypothèse de PAL

In [ ]:
rows = []
for pal in (1.45, 1.55, 1.75):
    C.GOLDBERG["pal"] = pal
    f = flag_underreporters(df.drop(columns=["bmr_kcal", "ei_bmr", "seuil_goldberg", "sous_declarant"]))
    f = f.dropna(subset=["sous_declarant"])
    rows.append({"PAL": pal, "seuil_3_jours": goldberg_cutoff(3),
                 "% sous-déclarants": 100 * weighted_mean(f.sous_declarant.astype(float), f.poids_enquete)})
C.GOLDBERG["pal"] = 1.55          # on remet la valeur par défaut
pd.DataFrame(rows).set_index("PAL")

## 4. Pondéré vs non pondéré

In [ ]:
rows = []
for n in NUTS:
    w = describe_weighted(df, n).iloc[0]
    u = describe_weighted(df.assign(_w=1.0), n, weight="_w").iloc[0]
    rows.append({"nutriment": C.NUTRIENTS[n][0], "pondere": w.moyenne, "non_pondere": u.moyenne,
                 "ecart_%": 100 * (u.moyenne / w.moyenne - 1), "largeur_IC_pondere": w.ic95_haut - w.ic95_bas,
                 "largeur_IC_non_pondere": u.ic95_haut - u.ic95_bas})
pd.DataFrame(rows).set_index("nutriment")

**Lecture** — L'écart pondéré / non pondéré mesure le **biais** qu'on commettrait en ignorant le plan de sondage (si un groupe est sur-représenté dans l'échantillon). Les IC pondérés sont plus larges : la dispersion des poids réduit la taille d'échantillon effective (Kish, notebook 02).

## 5. Intervalles de confiance : linéarisation vs bootstrap

Le bootstrap rééchantillonne les **individus** (avec leurs poids) 1 000 fois et recalcule la moyenne pondérée ; l'IC est donné par les percentiles 2,5 % et 97,5 %. Si les deux méthodes concordent, la formule analytique est validée.

In [ ]:
rng = np.random.default_rng(2024)
x, w = df["energie_kcal"].to_numpy(), df["poids_enquete"].to_numpy()
boot = np.array([weighted_mean(x[i], w[i]) for i in (rng.integers(0, len(x), len(x)) for _ in range(1000))])
m, se = weighted_mean(x, w), weighted_se(x, w)
pd.DataFrame({"moyenne": [m, m], "ic95_bas": [m - 1.96 * se, np.percentile(boot, 2.5)],
              "ic95_haut": [m + 1.96 * se, np.percentile(boot, 97.5)]},
             index=["linéarisation", "bootstrap (B = 1000)"])

## Conclusion de robustesse

| Hypothèse testée | Effet sur les résultats | Conclusion |
|---|---|---|
| Exclusion des sous-déclarants | apports absolus en hausse, parts et densités stables | présenter les deux versions pour Q1-Q3 |
| PAL 1,45 → 1,75 | la prévalence de sous-déclaration varie nettement | le chiffre de prévalence dépend d'une hypothèse : le dire |
| Pondérations | écarts modérés, IC plus larges | toujours pondérer |
| Méthode d'IC | linéarisation ≈ bootstrap | formule analytique validée |

*(Tableau à mettre à jour avec les valeurs obtenues sur les vraies données.)*

➡️ Suite (bonus) : `06_bonus_profils_inca2_mapping.ipynb`